In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

folder_path = '/content/drive/MyDrive/Хакатон'
if os.path.exists(folder_path):
    for file_name in os.listdir(folder_path):
        print(file_name)
else:
    print(f"Folder not found: {folder_path}")

УДО.csv
Rus_schools_final.csv


In [3]:
import pandas as pd

folder_path = '/content/drive/MyDrive/Хакатон'
schools_path = os.path.join(folder_path, 'Rus_schools_final.csv')
udo_path = os.path.join(folder_path, 'УДО.csv')

try:
    df_schools = pd.read_csv(schools_path, encoding="cp1251")
    print("Successfully loaded Rus_schools_final.csv")
    display(df_schools.head())
except FileNotFoundError:
    print(f"Error: {schools_path} not found.")

try:
    df_udo = pd.read_csv(udo_path, encoding="utf-8", sep=';')
    print("Successfully loaded УДО.csv")
    display(df_udo.head())
except FileNotFoundError:
    print(f"Error: {udo_path} not found.")

Successfully loaded Rus_schools_final.csv


,Unnamed: 0,name,struct,addr,lat,lon
0,0,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"420087, Республика Татарстан, г. Казань, ул.Ри...",55.76370,49.18172
1,1,"Муниципальная образовательная школа-интернат ""...",(Муниципальное образовательное учреждение),"420103, Республика Татарстан, г. Казань, ул.Че...",55.82440,49.12312
2,2,Муниципальное учреждение образования для детей...,(Муниципальное образовательное учреждение),"420100, Республика Татарстан, г. Казань, ул. Ю...",55.74462,49.20560
3,3,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"420011, Республика Татарстан, г. Казань, Ферма-2",55.71716,49.16222
4,4,Специальное (коррекционное)образовательное учр...,(Государственное образовательное учреждение),"420036, Республика Татарстан, г. Казань, ул.Ти...",55.84911,49.07660


Successfully loaded УДО.csv


,name,addr,lat,lon,Unnamed: 4
0,МАУ ДО «Военно-патриотический парк «Патриот»,"654018, Октябрьский проспект, 28",53.750985,87.145392,NaN
1,МАУ ДО «Военно-патриотический парк «Патриот»,"654101, Абагурское шоссе, 10",53.740551,87.207544,NaN
2,МБ ОУ ДО «Городской Дворец детского (юношеског...,"654018, Улица Циолковского, 78а",53.754012,87.151178,NaN
3,МБ ОУ ДО «Городской Дворец детского (юношеског...,"654041,Проспект Бардина, 5",53.749556,87.120077,NaN
4,МАУ ДО «Детско-юношеский центр «Орион»,"654079, Улица Кутузова, 5а",53.752563,87.116245,NaN


In [4]:
df_schools = df_schools[df_schools['addr'].str.contains('г. Новокузнецк', case=False, na=False)]
display(df_schools.head())

,Unnamed: 0,name,struct,addr,lat,lon
16551,16551,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"654044, Кемеровская область, г. Новокузнецк, у...",53.89191,87.12040
16553,16553,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"654000, Кемеровская область, г. Новокузнецк, у...",53.74954,87.11366
24634,24634,Муниципальное общеобразовательное учреждение С...,(Муниципальное образовательное учреждение),"654041, Кемеровская область, г. Новокузнецк, у...",53.74584,87.12456
24636,24636,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"654034, Кемеровская область, г. Новокузнецк, у...",53.79304,87.23308
24637,24637,"Муниципальное общеобразовательное учреждение ""...",(Муниципальное образовательное учреждение),"654034, Кемеровская область, г. Новокузнецк, у...",53.79060,87.22340


In [5]:
#@title Параметры для анализа
radius_km = 3.0  #@param {type:"number", min:0.5, max:10, step:0.5}

In [6]:
import folium
from geopy.distance import geodesic

# Подсчёт количества УДО в радиусе 3 км для каждой школы
udo_counts = []
for _, school in df_schools.iterrows():
    school_coord = (school['lat'], school['lon'])
    count = 0
    for _, udo in df_udo.iterrows():
        udo_coord = (udo['lat'], udo['lon'])
        if geodesic(school_coord, udo_coord).km <= radius_km:
            count += 1
    udo_counts.append(count)

df_schools['udo_count'] = udo_counts

# Создание карты
center_lat = df_schools['lat'].mean()
center_lon = df_schools['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Добавляем школы
for _, row in df_schools.iterrows():
    color = 'blue' if row['udo_count'] > 0 else 'gray'
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"{row['name']}<br>УДО в радиусе {radius_km} км: {row['udo_count']}",
        icon=folium.Icon(color=color, icon='graduation-cap', prefix='fa')
    ).add_to(m)

    # Добавляем бледную полупрозрачную окружность для школ без УДО
    if row['udo_count'] == 0:
        folium.Circle(
            location=[row['lat'], row['lon']],
            radius=radius_km * 1000,  # переводим км в метры
            color='gray',
            fill=True,
            fill_opacity=0.1,
            weight=1
        ).add_to(m)

# Добавляем УДО
for _, row in df_udo.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['name'],
        icon=folium.Icon(color='red', icon='child', prefix='fa')
    ).add_to(m)

In [7]:
m

In [8]:
import plotly.graph_objects as go

# Pie chart: школы с УДО и без
with_udo = (df_schools['udo_count'] > 0).sum()
without_udo = (df_schools['udo_count'] == 0).sum()

pie = go.Figure(data=[go.Pie(
    labels=['Школы с УДО', 'Школы без УДО'],
    values=[with_udo, without_udo],
    hole=0.3,
    marker=dict(colors=['skyblue', 'lightgray']),
    hoverinfo='label+percent+value'
)])
pie.update_layout(title_text='Доля школ с УДО и без УДО')

# Bar chart: распределение школ с УДО по количеству
udo_distribution = df_schools[df_schools['udo_count'] > 0]['udo_count'].value_counts().sort_index()

bar = go.Figure(data=[go.Bar(
    x=udo_distribution.index,
    y=udo_distribution.values,
    marker_color='skyblue'
)])
bar.update_layout(
    title='Распределение школ с УДО по количеству',
    xaxis_title=f'Количество УДО в радиусе {radius_km} км',
    yaxis_title='Количество школ'
)

# Отображение графиков
pie.show()
bar.show()